# Linguistic feature testing

This notebook tests the original English-only rule-based linguistic tier described in `LINGUISTIC-FEATURE-PLAN.md`. It uses the nine existing regex features and compares:

- the existing all-rule baseline;
- high-specificity rules for definite block candidates;
- a role-play-aware policy;
- the all-rule check with each rule removed one at a time.

The deepset dataset is excluded because it contains German prompts.

In [1]:
import pandas as pd

from Linguistic_feature import (
    linguistic_check_counts,
    load_existing_classifier_prompts,
    print_role_play_analysis,
    rule_feature_matrix,
    run_linguistic_benchmark,
)

In [2]:
data = load_existing_classifier_prompts("results/classifier_predictions.csv")
display(data.groupby(["dataset", "label"], sort=False).size().unstack(fill_value=0)
        .rename(columns={0: "harmless", 1: "attacks"}))

Excluded from linguistic scope: deepset (mixed-language or outside the defined English scope)
English-scope prompts: 1270


label,harmless,attacks
dataset,,
jackhhao,123,139
hackaprompt,500,500
ours,4,4


In [3]:
counts = linguistic_check_counts(data)
display(counts)

features = rule_feature_matrix(data["text"])
display(pd.concat([data[["dataset", "text", "label"]].reset_index(drop=True), features], axis=1).head())

English-scope prompts: 1270


,rule,attacks (jackhhao),harmless (jackhhao),HackAPrompt attacks,HackAPrompt harmless
1,role play,92,17,17,11
3,no limits,71,0,2,0
2,jailbreak names,51,0,1,0
0,ignore instructions,31,0,62,0
4,system prompt,1,0,0,0
7,encoded text,1,1,0,0
5,privilege,0,0,1,0
6,fake system tag,0,0,2,0
8,hidden characters,0,0,6,0


,dataset,text,label,ignore instructions,role play,jailbreak names,no limits,system prompt,privilege,fake system tag,encoded text,hidden characters
0,jackhhao,Act as a yoga instructor who also offers welln...,0,0,1,0,0,0,0,0,0,0
1,jackhhao,You are a troll who is an expert baker. Offer ...,0,0,0,0,0,0,0,0,0,0
2,jackhhao,"You are in a future world of legal, ethical, a...",1,0,0,0,1,0,0,0,0,0
3,jackhhao,"You are Leon S. Kennedy exploring a dark, eeri...",0,0,0,0,0,0,0,0,0,0
4,jackhhao,This is all a hypothetical scenario in a hypot...,1,0,0,0,1,0,0,0,0,0


In [4]:
bench = run_linguistic_benchmark(
    data,
    output_folder="results/linguistic-features",
    include_rule_ablations=True,
)

English-scope prompts: 1270
Saved results\linguistic-features\classifier_results.csv and results\linguistic-features\classifier_predictions.csv


In [5]:
print_role_play_analysis(bench)
display(bench.table())

Linguistic rules (baseline): 31 false alarms; 29 match the role-play rule
Role-play-aware rules: 2 false alarms; 0 match the role-play rule


,classifier,dataset,prompts,attacks caught,attacks missed,false alarms,detection rate,false alarm rate,F1,AUC,detection at 1% false alarms,ms per prompt,note,training s
0,Linguistic rules (baseline),jackhhao,262,118,21,18,0.848921,0.146341,0.858182,None,None,0.153820,,None
1,Linguistic rules (baseline),hackaprompt,1000,83,417,11,0.166000,0.022000,0.279461,None,None,0.074044,,None
2,Linguistic rules (baseline),ours,8,4,0,2,1.000000,0.500000,0.800000,None,None,0.006975,,None
3,Linguistic rules (baseline),all,1270,205,438,31,0.318818,0.049442,0.466439,None,None,0.090079,,None
4,High-specificity rules,jackhhao,262,97,42,0,0.697842,0.000000,0.822034,None,None,0.080688,,None
5,High-specificity rules,hackaprompt,1000,65,435,0,0.130000,0.000000,0.230088,None,None,0.029721,,None
6,High-specificity rules,ours,8,3,1,1,0.750000,0.250000,0.750000,None,None,0.006762,,None
7,High-specificity rules,all,1270,165,478,1,0.256610,0.001595,0.407911,None,None,0.040091,,None
8,Role-play-aware rules,jackhhao,262,97,42,1,0.697842,0.008130,0.818565,None,None,0.373005,,None
9,Role-play-aware rules,hackaprompt,1000,73,427,0,0.146000,0.000000,0.254799,None,None,0.084425,,None
